In [ ]:
import os
import random
import sys

import numpy as np
import pandas as pd
import torch
from Bio import SeqIO
from rdkit import Chem
from rdkit.Chem import AllChem

CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
sys.path.append("../../tools/")
from distinct import remove_duplicates

In [ ]:
remove_duplicates("../../enzyme/UGT_enzyme.fasta", "../../enzyme/UGT_enzyme_q.fasta")
! python {CURRENT_DIR}/../../tools/extract.py esm1b_t33_650M_UR50S  {CURRENT_DIR}/../../enzyme/UGT_enzyme_q.fasta  {CURRENT_DIR}/../../enzyme/esm_ugt --repr_layers 33 --include mean

In [ ]:
fasta_file = CURRENT_DIR + "/../../enzyme/UGT_enzyme_q.fasta"
sequences = []
for record in SeqIO.parse(fasta_file, "fasta"):
    sequences.append([record.id, str(record.seq)])
ourdata = pd.DataFrame(sequences, columns=["enzyme", "Sequence"])

In [ ]:
ourdata["ESM1b"] = ""
rep_dict = CURRENT_DIR + "/../../enzyme/esm_ugt/"
print(rep_dict)
for ind in ourdata.index:
    try:
        esms = torch.load(rep_dict + str(ourdata["enzyme"][ind]) + ".pt")
        ourdata["ESM1b"][ind] = esms["mean_representations"][33].tolist()
    except:
        print(str(ourdata["enzyme"][ind]))

In [ ]:
subdata = pd.read_excel(CURRENT_DIR + "/../data/UGT_Substrate.xlsx")

In [ ]:
ourdata = pd.merge(ourdata, subdata, how="left", left_on="enzyme", right_on="UGT ID")
ourdata.rename(columns={"UGT ID": "substrate"}, inplace=True)

In [ ]:
ourdata["Binding"] = 1

In [ ]:
ourdata["ECFP"] = ""
for ind in ourdata.index:
    name = ourdata.at[ind, "enzyme"]
    try:
        ecfps = Chem.MolFromMolFile(
            CURRENT_DIR + "/../../substrate/UGTSubstrate/" + name + ".sdf"
        )
        ecfpso = AllChem.GetMorganFingerprintAsBitVect(
            ecfps, radius=2, nBits=1024
        ).ToBitString()
        ourdata.loc[ind, "ECFP"] = ecfpso
    except:
        print("no this name sdf:", name)

In [ ]:
ourdata = ourdata[
    (ourdata["ESM1b"] != "")
    & (ourdata["enzyme"] != "")
    & (ourdata["Sequence"] != "")
    & (ourdata["ECFP"] != "")
    & (ourdata["substrate"] != "")
].dropna()

In [ ]:
ourdata

In [ ]:
has_null_values = ourdata.isna().any().any()
has_empty_strings = ourdata.applymap(lambda x: x == "")
if has_empty_strings.any().any():
    print("DataFrame contains empty strings.")
if has_null_values:
    print("DataFrame contains empty values.")

In [ ]:
enzyme_unique_count = ourdata["enzyme"].nunique()
substrate_unique_count = ourdata["substrate"].nunique()
ecfp_unique_count = ourdata["ECFP"].nunique()

In [ ]:
all_substrate = ourdata["substrate"].unique()

enzyme_unique = ourdata["enzyme"].unique()

ourdata_neg = pd.DataFrame(columns=["enzyme", "substrate", "Sequence", "ESM1b"])
rows_to_add = []
for enzyme in enzyme_unique:
    selected_substrates = random.sample(list(all_substrate), 2)
    rows_to_add.append({"enzyme": enzyme, "substrate": selected_substrates[0]})
ourdata_neg = pd.concat([ourdata_neg, pd.DataFrame(rows_to_add)], ignore_index=True)

for index, row in ourdata.iterrows():
    matching_rows = ourdata_neg[(ourdata_neg["enzyme"] == row["enzyme"])]
    if not matching_rows.empty:
        for matching_index in matching_rows.index:
            ourdata_neg.at[matching_index, "ESM1b"] = row["ESM1b"]
for index, row in ourdata.iterrows():
    matching_rows = ourdata_neg[(ourdata_neg["substrate"] == row["substrate"])]
    if not matching_rows.empty:
        for matching_index in matching_rows.index:
            ourdata_neg.at[matching_index, "ECFP"] = row["ECFP"]
ourdata_neg["Binding"] = 0

merged_data = pd.concat([ourdata, ourdata_neg], ignore_index=True)
selected_columns = ["enzyme", "substrate", "Binding", "ECFP", "ESM1b"]
final_data = merged_data.loc[:, selected_columns]
final_data.to_pickle(CURRENT_DIR + "/../data/trainUGT.pkl")

In [ ]:
final_data